# Temporal: Durable Workflows for Agent Orchestration

This notebook outlines **conceptual patterns** for building long-running, fault-tolerant agent workflows with [Temporal](https://temporal.io/). A running **Temporal server** (self-hosted or Temporal Cloud) and worker process are required to execute the code below; the snippets illustrate API shape and best practices.

## Introduction: why durability matters for agents

Agent runs often span many LLM calls, tool invocations, and persistence steps. Without durable orchestration, a process crash or deploy mid-run loses in-flight state. Temporal records workflow history and **replays** deterministic workflow code after failures, while **activities** execute side effects (network I/O, tools) with configurable retries. That separation—deterministic workflow vs. idempotent or retriable activities—is the core design lever for reliable agents.

In [ ]:
try:
    from temporalio import activity, workflow
    from temporalio.common import RetryPolicy
    TEMPORAL_AVAILABLE = True
except ImportError:
    TEMPORAL_AVAILABLE = False
    print("Install optional dependency: pip install 'agentic-assistants[durable]' or temporalio")

# Pattern sketch: activities are short-lived side effects (LLM, tools, storage).
# Uncomment and run inside a Temporal worker after installing temporalio.

if TEMPORAL_AVAILABLE:
    @activity.defn
    async def call_llm_activity(prompt: str) -> str:
        """Wrap your LLM client; keep timeouts and idempotency keys in mind."""
        return f"[synthetic completion for: {prompt[:40]}...]"

    @activity.defn
    async def run_tool_activity(tool_name: str, args: dict) -> dict:
        return {"tool": tool_name, "result": "ok", "args_echo": args}

    @activity.defn
    async def persist_memory_activity(session_id: str, blob: str) -> None:
        # Write to DB, object store, or vector index
        pass

    print("Activity stubs defined (LLM, tool, memory persistence).")
else:
    print("Skipping activity definitions (temporalio not installed).")

## Workflow: multi-step agent reasoning

Workflow code must be **deterministic** (no direct `random`, wall-clock, or network). Delegate non-determinism to activities via `workflow.execute_activity`. Chain plan → act → reflect steps as separate activities so each can have its own timeout and retry policy.

In [ ]:
from datetime import timedelta

if TEMPORAL_AVAILABLE:
    @workflow.defn
    class AgentReasoningWorkflow:
        @workflow.run
        async def run(self, user_goal: str) -> str:
            plan = await workflow.execute_activity(
                call_llm_activity,
                f"Plan steps for: {user_goal}",
                start_to_close_timeout=timedelta(seconds=60),
            )
            tool_out = await workflow.execute_activity(
                run_tool_activity,
                args=["search", {"query": user_goal}],
                start_to_close_timeout=timedelta(seconds=30),
            )
            answer = await workflow.execute_activity(
                call_llm_activity,
                f"Synthesize: plan={plan!r} tool={tool_out!r}",
                start_to_close_timeout=timedelta(seconds=60),
            )
            await workflow.execute_activity(
                persist_memory_activity,
                args=[workflow.info().workflow_id, answer],
                start_to_close_timeout=timedelta(seconds=15),
            )
            return answer
    print("AgentReasoningWorkflow class defined.")
else:
    print("Define workflows after installing temporalio; import timedelta inside workflow module in production.")

The workflow module should import `timedelta` in a way that satisfies Temporal's workflow sandbox rules (see Temporal Python SDK docs). Avoid non-deterministic calls inside workflow code.

In [ ]:
from datetime import timedelta

if TEMPORAL_AVAILABLE:
    # Example retry policy for flaky LLM / HTTP calls
    llm_retry = RetryPolicy(
        initial_interval=timedelta(seconds=1),
        maximum_interval=timedelta(seconds=30),
        backoff_coefficient=2.0,
        maximum_attempts=5,
    )
    print("RetryPolicy example:", llm_retry)
    # In execute_activity: retry_policy=llm_retry
    # Recovery patterns: catch activity failures in workflow, branch to compensations,
    # or use Saga-style child workflows to undo partial work.
else:
    print("RetryPolicy example skipped without temporalio.")

## Retry policies and failure recovery

- Use **shorter** timeouts and **more** retries for idempotent reads; **fewer** retries for billing or non-idempotent writes unless deduplicated.
- **Child workflows** isolate sub-goals (e.g., per-document processing) so failure domains stay small.
- **Parallel execution**: start multiple child workflows or activities with `asyncio.gather`-style patterns the SDK supports (e.g., `workflow.execute_activity` in parallel tasks).

In [ ]:
if TEMPORAL_AVAILABLE:
    @workflow.defn
    class ParentOrchestrator:
        @workflow.run
        async def run(self, tasks: list[str]) -> list[str]:
            # Conceptual: execute child workflows in parallel for independent subtasks
            handles = []
            for i, t in enumerate(tasks):
                h = await workflow.start_child_workflow(
                    AgentReasoningWorkflow.run,
                    t,
                    id=f"child-{workflow.info().workflow_id}-{i}",
                )
                handles.append(h)
            return [await h for h in handles]
    print("ParentOrchestrator with child workflows defined.")
else:
    print("Child workflow pattern: start_child_workflow + await result.")

## Conclusion and architecture recommendations

1. **Model each tool and LLM call as an activity** with explicit timeouts and retries; keep workflow code linear and replay-safe.
2. **Pass correlation IDs** (workflow run ID, trace context) into activities for observability.
3. **Use child workflows** when sub-runs need independent lifecycle, policies, or team ownership.
4. **Run a dedicated worker fleet** connected to your Temporal namespace; notebooks are for design only unless you start a dev server (`temporal server start-dev`) and a worker process.

This keeps agent orchestration **durable**, **observable**, and **evolvable** without custom checkpoint stores for every framework.